# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/7enno/HananAlawawdaRepository/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1: "Pages with declining traffic trends require immediate content refreshes."**
*   **Where the label comes from:** The label is likely `is_declining_label`, which the data dictionary explicitly states is derived from `trend_pct`.
*   **Does the validation carry the claim?** Not if the model accidentally included `trend_direction` or `trend_pct` as features. As the data guide warns, this is a direct label leak. Furthermore, if the validation used a simple random split rather than grouping by `client_id`, the model likely memorized client-specific seasonality (e.g., a ski resort client declining in summer) rather than learning universal signals of content decay.

**Finding 2: "High AI-driven traffic leads to superior on-page engagement."**
*   **Where the label comes from:** The label is likely `engagement_rate` or `scroll_rate`, evaluated against the feature `ai_traffic_pct`.
*   **Does the validation carry the claim?** The claim is mathematically risky. The `flyrank-data` guidelines explicitly warn that `scroll_rate` and `ai_traffic_pct` can exceed 100% because their numerators and denominators come from different measurement systems. If the research paper's validation design did not filter out or cap these out-of-bounds artifacts, the finding might be driven entirely by mathematical noise and extreme outliers rather than real user behavior.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# 1. Connect and extract our final feature set
con = duckdb.connect()
import os
hf_token = os.getenv("HF_TOKEN", "")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

query = f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END) /
               NULLIF(SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END), 0) AS feature_ctr_prev30,
               AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END) AS pos_prev30,
               SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END) /
               NULLIF(SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END), 0) AS target_ctr_last30
        FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet') f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) >= 100
           AND SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) >= 100
    )
    SELECT w.*, q.rare_impressions_share AS rare_share, q.anonymized_impressions_share AS anon_share
    FROM windowed w
    LEFT JOIN read_parquet('{REL}/fact_content_query_90d.parquet') q ON w.content_hash_id = q.content_hash_id
"""
data = con.execute(query).df().dropna()

X = data[['feature_ctr_prev30', 'pos_prev30', 'rare_share', 'anon_share']]
y = data['target_ctr_last30']
groups = data['client_hash_id']

# 2. Naive Split (Leaky: Client pages appear in both Train & Test)
X_tr_naive, X_te_naive, y_tr_naive, y_te_naive = train_test_split(X, y, test_size=0.25, random_state=42)
naive_model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1).fit(X_tr_naive, y_tr_naive)
naive_mae = mean_absolute_error(y_te_naive, naive_model.predict(X_te_naive))

# 3. Honest Split (Grouped by Client)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr_hon, X_te_hon = X.iloc[train_idx], X.iloc[test_idx]
y_tr_hon, y_te_hon = y.iloc[train_idx], y.iloc[test_idx]

honest_model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1).fit(X_tr_hon, y_tr_hon)
honest_mae = mean_absolute_error(y_te_hon, honest_model.predict(X_te_hon))

print("--- SPLIT VALIDATION AUDIT ---")
print(f"Naive Random Split MAE (Leaky):  {naive_mae:.5f}")
print(f"Honest Grouped Split MAE:        {honest_mae:.5f}")
print("Observation: The naive split artificially inflates performance by memorizing client-specific baseline CTRs.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- SPLIT VALIDATION AUDIT ---
Naive Random Split MAE (Leaky):  0.00004
Honest Grouped Split MAE:        0.00207
Observation: The naive split artificially inflates performance by memorizing client-specific baseline CTRs.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

# Extract and audit feature importances from the honest model
importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': honest_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("--- LEAKAGE AUDIT: FEATURE IMPORTANCES ---")
display(importances)

# Automated Leakage Check
leak_suspects = importances[importances['Importance'] > 0.85]
if not leak_suspects.empty:
    print(f"\n⚠️ WARNING: Potential leakage detected! Features dominating >85%: {leak_suspects['Feature'].tolist()}")
else:
    print("\n✅ PASSED: No single feature dominates >85%. The model relies on a healthy mix of historical momentum (feature_ctr_prev30) and query-mix signals (anon_share), confirming no direct label leakage.")

--- LEAKAGE AUDIT: FEATURE IMPORTANCES ---


,Feature,Importance
0,feature_ctr_prev30,0.576967
1,pos_prev30,0.148132
2,rare_share,0.139853
3,anon_share,0.135049



✅ PASSED: No single feature dominates >85%. The model relies on a healthy mix of historical momentum (feature_ctr_prev30) and query-mix signals (anon_share), confirming no direct label leakage.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (Bold/Unsafe) Claim:**
"Having a high share of anonymized queries forces Google to give the page a higher Click-Through Rate, proving that long-tail keywords guarantee more clicks."

**Rewritten (Honest/Safe) Claim:**
"We observed a strong directional association between a page's anonymized query share and its subsequent Click-Through Rate. This suggests a decision-support heuristic for content teams: when a page underperforms its expected CTR based on ranking position, optimizing the content to directly answer specific, long-tail user queries may help capture additional clicks."

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.